# MNIST Digit Classification

## Importing Code

The first thing we'll do is _import_ code that will help us demonstrate classification in a realistic setting. Most of this is detail you don't need to know in AI 109, but a few things are worth pointing out:

- The line `import torch` enables us to use the PyTorch library, which is an industrial-strength package for working with neural networks. It's used widely in industry.
- The lines that include `sklearn` let us use the Scikit-Learn library, which is another widely used library. Here we're using it for data visualization.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from matplotlib.colors import BoundaryNorm
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
torch.set_num_threads(4)

## Using a GPU

One of the things that distinguishes neural networks from other forms of machine learning is that it's typical to use GPUs when training large neural networks. This next cell tests if _CUDA_ is available. CUDA is the software that runs GPU code on Nvidia GPUs. 

I'm running this on a MacBook, which doesn't have CUDA. In the lab, you should see a device name like `"cuda"` or maybe `"cuda:0"`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")

## Getting Some Data: MNIST

The MNIST dataset is a collection of images of handwritten _digits_. Each image is 28 pixels by 28 pixels, and they're grayscale images (we'll look at an example below). 

MNIST is a good example of a _classification problem_: We have complicated inputs (images with 784 pixels) that we want to assign class labels to. We know we have 10 class labels (one per digit), so the space of possibilities isn't too small or too big. 

This next cell downloads the data, and sets up some code to help us load the data to train a perceptron.

In [ ]:
train = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST("data", train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train, batch_size=256, shuffle=True)
test_loader = DataLoader(test, batch_size=1024)

## Visualizing a Digit

Let's visualize the first example in the data set. You can change the 0 in the line `image, label = train[0]` to a different value to see different inputs.

In [ ]:
image, label = train[0]
plt.figure(figsize=(3, 3))
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Digit: {label}")
plt.axis("off")
plt.show()

Another way to visualize the digit is to just print it out. This is maybe hard to read, but if you look at it for a minute, the same pattern emerges. This is what a classifier actually "sees," so it's not a bad idea to look at data like this from time to time.

Notice that `train[0]` includes the image and also a correct _label_ for the image. We can use that label to improve how our model performs.

In [ ]:
torch.set_printoptions(linewidth=300, precision=2, sci_mode=False)
print(train[0])
torch.set_printoptions(profile="default")

It may be easier to see the pattern if you _round_ the data, which we can do simply:

In [ ]:
torch.set_printoptions(linewidth=300, precision=2, sci_mode=False)
print(train[0][0].round())
torch.set_printoptions(profile="default")

## Defining a Perceptron in Code

Well... Not quite a perceptron. Modern _neural networks_ differ from perceptrons in their use of _differentiable_ operations rather than hard 0-1 thresholds. But the object below is a reasonable approximation to the idea of a perceptron.

The `nn.Linear(784, 10)` line creates the _weights_ of the Perceptron for us. The two numbers tell us how many inputs there are to the perceptron (784), and how many outputs there are (10). This model outputs 10 numbers. We can think of the first output as the amount of confidence that the model has that the input was 0. Similarly, we can think of the 10th output as the amount of confidence that the input was a 9. The `fc1` in that line is short for `fully-connected layer`, because there's a weight for every connection between an input and an output. That means there are  `784*10 + 10 = 7850` weights and biases in this perceptron. Each of these weights can be adjusted up or down, and each such adjustment will change how much confidence the model assigns to a label. The "game" of machine learning is finding the weights that lead to the best performance.

The `forward` function is the place to look to see what the classifier _does_. In this case, the first thing it does is _flatten_ the image into a single list of 784 pixel values. Then it uses the perceptron weights to calculate a confidence score for each digit based on all the pixels in the input image. The technical term for such a score is a _logit_.

This kind of model is not widely used for image classification, but is a useful baseline to start with.

In [ ]:
class PerceptronClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)

    def forward(self, image):
        image = torch.flatten(image, start_dim=1)
        return self.fc1(image)

## Multi-Layer Perceptrons

One of the things that AI researchers discovered early on was that you could use the outputs of one perceptron as the inputs of another one. This idea is the basis of _deep learning_. The resulting classifier is called a _multi-layer perceptron_, or _MLP_. 

Here, we have an MLP with a single _hidden layer_ that sits between the input and the output. The first layer (look at `fc1`) has 784 inputs. But we don't have to limit ourselves to 10 outputs any more - we can use any size output that we want. We call that size `hidden_dim`. We would say that the hidden layer has `hidden_dim` _units_, each producing an _activation_. This number is determined when we create a new `MLP`.

One thing you might wonder as you read the code below is what `F.sigmoid` is doing. The _sigmoid_ function is called an _activation_ function. It compresses the output of a previous layer into a well-defined range of possible output values. The sigmoid function compresses values into the range $(0,1)$. You'll learn _why_ we do that in a future class.

This model has `784 * hidden_dim + hidden_dim * 10 + hidden_dim + 10` weights and biases that can be adjusted.

MLPs (or their close relative, the _mixture of experts layer_) are one of the core components of most AI chat systems in use today.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 10)

    def forward(self, image):
        image = torch.flatten(image, start_dim=1)
        activations1 = self.fc1(image)
        activations2 = F.sigmoid(activations1)
        return self.fc2(activations2)

## A Convolutional Neural Network

We won't walk through the details, but `ConvNet` below is an example of a _convolutional neural network_, which is a type of neural network that is optimized for processing image-like data. I'm including this as an example to show you a strong baseline for performance for classifying MNIST digits.

Convolutional networks use clever ideas to _share_ weights, and they explicitly take into account the two-dimensional structure of images (which the previous two models ignored). This type of network is extremely common in computer vision and image/video generation.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, hidden_dim)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(hidden_dim, 10)

    def forward(self, image):
        # Each pooling step halves the image dimensions: 28 -> 14 -> 7.
        features = self.pool(F.relu(self.conv1(image)))
        features = self.pool(F.relu(self.conv2(features)))
        features = torch.flatten(features, start_dim=1)
        features = self.dropout(F.relu(self.fc1(features)))
        return self.fc2(features)  # Raw logits for CrossEntropyLoss.

## Visualizing Weights Before and After Training

The function below draws the weights of the selected network. **Blue means negative, white means zero, and magenta means positive.** Stronger colors mean larger magnitudes. Each layer has its own symmetric color scale, which is recalculated each time we draw it. Check the numbers on the color bars when comparing layers or comparing weights before and after training: the same color can represent different values.

- For a fully connected (`Linear`) layer, rows are output units and columns are input units. Each square is one weight. Wide matrices appear thin because the squares keep their true proportions.
- For a convolutional (`Conv2d`) layer, each small rectangle is a filter (3 by 3 in this notebook). Filter rows correspond to output channels and filter columns to input channels. Gray lines separate filters.
- Biases are not shown. Pooling, activation, and dropout operations have no weight matrices to draw.

Run the next cell to define the drawing function; it does not create or train a model. After we create a classifier below, we call `plot_model_weights(model)` to see its random initial weights. Later, after training, we call the same function again to see how those weights have changed. You do not need to understand the drawing code to use it. The function works on CPU or GPU without changing the model.


In [ ]:
def plot_model_weights(model: nn.Module):
    """Display Linear matrices and tiled Conv2d filters; return the figures.

    Cells are square and colors are scaled separately for each layer.
    No weights are averaged or discarded. Biases are excluded.
    """
    from matplotlib.colors import LinearSegmentedColormap, Normalize

    cmap = LinearSegmentedColormap.from_list(
        "blue_white_magenta", ["#0000ff", "#ffffff", "#ff00ff"], N=257
    )
    figures = []
    for name, layer in model.named_modules():
        if not isinstance(layer, (nn.Linear, nn.Conv2d)):
            continue
        weights = layer.weight.detach().cpu().float().numpy()
        layer_name = name or type(layer).__name__
        if isinstance(layer, nn.Linear):
            matrix = weights
            xlabel, ylabel = "Input unit", "Output unit"
            description = f"{weights.shape[0]} outputs × {weights.shape[1]} inputs"
        else:
            outputs, inputs, height, width = weights.shape
            # Keep each spatial filter intact, then tile by channel pair.
            matrix = weights.transpose(0, 2, 1, 3).reshape(
                outputs * height, inputs * width
            )
            xlabel = "Input channel" if layer.groups == 1 else "Input channel within group"
            ylabel = "Output channel"
            description = f"{outputs} × {inputs} filters, each {height} × {width}"

        limit = float(np.abs(weights).max())
        # An all-zero layer is white, with a nondegenerate color scale.
        norm = Normalize(vmin=-(limit or 1.0), vmax=(limit or 1.0))
        rows, columns = matrix.shape
        figure_height = min(14, max(3, 12 * rows / columns + 1.5))
        figure_width = min(14, max(6, 12 * columns / rows + 2))
        fig, ax = plt.subplots(figsize=(figure_width, figure_height), layout="constrained")
        im = ax.imshow(matrix, cmap=cmap, norm=norm, interpolation="nearest", aspect="equal")
        fig.suptitle(f"{layer_name}.weight\n{description}")
        ax.set(xlabel=xlabel, ylabel=ylabel)

        if isinstance(layer, nn.Conv2d):
            for x in range(width, columns, width):
                ax.axvline(x - 0.5, color="gray", linewidth=0.4)
            for y in range(height, rows, height):
                ax.axhline(y - 0.5, color="gray", linewidth=0.4)
            # Label channel centers, thinning labels on large grids.
            input_ticks = np.arange(0, inputs, max(1, (inputs + 7) // 8))
            output_ticks = np.arange(0, outputs, max(1, (outputs + 7) // 8))
            ax.set_xticks(input_ticks * width + (width - 1) / 2, input_ticks)
            ax.set_yticks(output_ticks * height + (height - 1) / 2, output_ticks)

        orientation = "horizontal" if rows > columns else "vertical"
        fig.colorbar(im, ax=ax, orientation=orientation, shrink=0.8, pad=0.02, label="Weight value")
        figures.append(fig)
        plt.show()

    if not figures:
        raise ValueError("The model has no Linear or Conv2d layers to visualize.")
    return figures


## Creating a Classifier

The cells above defined _classes_, which are used as templates for classifiers. Now that we have some templates, we can use them to create classifiers that actually do something. 

Look at the code below. Any code that starts with a `#` is called a _comment_, and gets ignored by Python. So the cell creates a `PerceptronClassifier` and calls it `model`. It then _moves_ the model to the GPU if one is available. (If a GPU is not available, then the `model = model.to(device)` step does nothing.)

You can comment out the first line and uncomment one of the commented lines to choose a different classifier.

In [ ]:
#model = PerceptronClassifier()
#model = MLP(128)
model = ConvNet(128)
model = model.to(device)

### The Untrained Weights

The classifier has just been created, so these weights are random initial values, not values learned from handwritten digits. Run the next cell and look at the patterns and the ranges on the color bars. We will compare these pictures with the weights after training.


In [ ]:
weight_figures = plot_model_weights(model)

## Defining a Loss and Optimizer

We want to measure how well our classifiers perform, and then use those measurements to adjust the weights of our models. The two lines in the cell below define the two objects we need to do those tasks.

### Loss Functions

The way to think about performance measurement for classification problems is in terms of penalties, which we call _losses_. When the model does well and produces the right answer, it should get a small loss. Bigger losses should correspond to worse performance.  

You might think that the obvious way to measure performance is to say that if the model gets the right answer then give it a loss of 0, otherwise give it a loss of 1. This corresponds to measuring the _error rate_ of the model. For reasons you'll learn about in a future class, it's better to look at all 10 outputs of the model, and use those together in an expression called the _cross-entropy loss_.

### Optimizers

Once we can measure loss, we need a way to adjust the parameters to make the loss as small as possible. This process is called _optimization_, and the object that performs optimization is called an _optimizer_. We tell the optimizer which weights or _parameters_ we want to optimize, and we give it something called a _learning rate_ to control how fast it adjusts those parameters. 

The simplest optimizer is called _stochastic gradient descent_, or _SGD_. Its performance may not be great, so you could try the _Adam_ optimizer instead. Adam stands for _adaptive moment estimation_. You'll learn what that means in a future class. In either case, you can try making `lr` bigger or smaller and see what happens.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

#optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Looking at a Prediction

Before we train our model, we can ask it to look at an input and make a _prediction_ about what the value should be. Of course, the prediction will probably be wrong, but we'll fix that in a minute. In the next cell, we print out the 10 confidence values and call the prediction of the model the index of the largest confidence (starting with index 0):

In [ ]:
with torch.no_grad():
    model.eval()
    untrained_logits = model(train[0][0].unsqueeze(0).to(device))
    print(f"Untrained logits: {untrained_logits.data[0]}")
    print(f"Predicted label: {untrained_logits.argmax()}")

## The Training Loop

The model we created above starts with random values for its weights. We use our `loss_fn` and `optimizer` to _train_ our model to produce correct outputs. The next cell implements what we call the _training loop_. A training loop consists of a number of _epochs_, or passes through the data. By setting `epochs = 10`, we're saying we want to look at each example 10 times. 

The training loop looks at small _batches_ of images along with their corresponding labels, and repeats the following:

- Move the images and labels to the GPU (if necessary)
- Clear the gradients (you'll learn about this in a future class)
- Have the model produce _predictions_ for each input image
- Use `loss_fn` to compare each prediction to the correct label, producing a loss.
- _Backpropagate_ the errors from the loss into the model. This line calculates a collection of values that are used by the optimizer to update the model. That collection of values is called the  _gradient_. Calculating the gradient doesn't actually update the model though.
- Use the gradients calculated by `loss.backward()` to update the model.

The code prints the average loss per example as it goes. This number should go down. If it does, we say that the model is _learning_. 

In [ ]:
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(images)
        loss = loss_fn(predictions, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train):.4f}")

## Calculate Accuracy, Test a Prediction

We can calculate the accuracy of our _trained_ model, and see if it produces a correct prediction now:

In [ ]:
model.eval()
with torch.no_grad():
    correct = sum((model(images.to(device)).argmax(1) == labels.to(device)).sum().item() for images, labels in test_loader)
print(f"Test accuracy: {correct / len(test):.2%}")

In [ ]:
with torch.no_grad():
    model.eval()
    trained_logits = model(train[0][0].unsqueeze(0).to(device))
    print(f"Trained logits: {trained_logits.data[0]}")
    print(f"Predicted label: {trained_logits.argmax()}")

### The Trained Weights

Run the next cell to draw the weights of the model we just trained. Compare these pictures with the untrained weights above. What patterns have appeared or changed? How have the ranges on the color bars changed? Remember that the color scales are recalculated for each plot, so compare the numbers as well as the colors.

To inspect the trained weights again, rerun just this plotting cell. Rerunning the cell under "Creating a Classifier" would replace the trained model with a new, untrained one.


In [ ]:
weight_figures = plot_model_weights(model)

## Visualization

We can see how our classifier performs by _visualizing_ our data. We do this in two ways.

First, we perform _dimensionality reduction_ by _projecting_ each image onto a two-dimensional plane, and then we draw that. We also draw the _decision boundaries_ produced by the classifier on that plane. This kind of projection is useful, but imperfect.

Second, we show the _confusion matrix_ for our data. This is a table of numbers where the row tells us the correct answer and the column tells us the predicted answer. The number in cell i,j tells us how many examples had correct label i but predicted label j. This can give some insight into the types of errors made by our classifier.

You'll want to ignore all of the details of the code below - just look at the pictures.

In [ ]:
pca = PCA(n_components=2, svd_solver="randomized", random_state=0)
pca.fit(train.data[:10000].reshape(-1, 784).numpy().astype(np.float32) / 255)
points = pca.transform(test.data[:2000].reshape(-1, 784).numpy().astype(np.float32) / 255)
labels = test.targets[:2000].numpy()
x_min, y_min = points.min(axis=0) - 1
x_max, y_max = points.max(axis=0) + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
plane_images = torch.from_numpy(pca.inverse_transform(np.c_[xx.ravel(), yy.ravel()]).astype(np.float32))
with torch.no_grad():
    regions = torch.cat([model(batch.reshape(-1, 1, 28, 28).to(device)).argmax(1).cpu() for batch in plane_images.split(1024)]).numpy().reshape(xx.shape)

cmap = plt.get_cmap("tab10")
norm = BoundaryNorm(np.arange(-0.5, 10.5), cmap.N)
fig, ax = plt.subplots(figsize=(10, 8))
ax.contourf(xx, yy, regions, levels=np.arange(-0.5, 10.5), cmap=cmap, norm=norm, alpha=0.25)
for digit in np.unique(regions):
    mask = (regions == digit).astype(float)
    if mask.min() != mask.max():
        ax.contour(xx, yy, mask, levels=[0.5], colors="gray", linewidths=0.5)
scatter = ax.scatter(points[:, 0], points[:, 1], c=labels, cmap=cmap, norm=norm, s=10, alpha=0.7)
fig.colorbar(scatter, ax=ax, ticks=range(10), label="True digit")
ax.set(xlabel="Principal component 1", ylabel="Principal component 2",
       title="MNIST projections and model decision regions on the PCA plane")
plt.show()

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        y_true.append(labels.cpu())
        y_pred.append(model(images.to(device)).argmax(1).cpu())

cm = confusion_matrix(torch.cat(y_true).numpy(), torch.cat(y_pred).numpy(), labels=range(10))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
ax.set(title="MNIST test-set confusion matrix", xlabel="Predicted digit", ylabel="True digit")
plt.tight_layout()
plt.show()